In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import pandas as pd, numpy as np
from src.value import Params, add_value

sc = pd.read_parquet("../data/scored.parquet")
P = Params()
v = add_value(sc, P)

# 1) Cek dengan hitungan tangan (contoh di 13a)
contoh = pd.DataFrame({"monthly_charge": [46_000.0], "p_churn": [0.5]})
c = add_value(contoh, P).iloc[0]
print(c[["clv", "saved_value", "ev", "break_even_p"]].round(3).to_dict())
assert abs(c["clv"] - 220_800) < 1e-6
assert abs(c["ev"] - 22_600) < 1e-6
assert abs(c["break_even_p"] - 5_000 / (0.25 * 220_800)) < 1e-9

{'clv': 220800.0, 'saved_value': 27600.0, 'ev': 22600.0, 'break_even_p': 0.091}


In [2]:
# 2) Gambaran umum
print(v[["clv", "ev", "break_even_p"]].describe().round(3))
n_pos = int((v.ev > 0).sum())
print(f"\nEV > 0: {n_pos} dari {len(v)} ({n_pos/len(v):.1%})")
print("Biaya jika SEMUA dihubungi : Rp", f"{len(v) * P.eff_cost:,.0f}")
print("Biaya jika hanya EV > 0    : Rp", f"{n_pos * P.eff_cost:,.0f}")
print("Total untung bersih (EV>0) : Rp", f"{v.loc[v.ev > 0, 'ev'].sum():,.0f}")

              clv         ev  break_even_p
count    7043.000   7043.000      7043.000
mean   220800.000  11832.002         0.128
std    102589.697  18282.208         0.087
min     62221.969  -4853.075         0.049
25%    121034.514  -3018.254         0.065
50%    239852.904   3228.423         0.083
75%    306336.651  23212.649         0.165
max    404868.974  72937.536         0.321

EV > 0: 4195 dari 7043 (59.6%)
Biaya jika SEMUA dihubungi : Rp 35,215,000
Biaya jika hanya EV > 0    : Rp 20,975,000
Total untung bersih (EV>0) : Rp 92,367,154


In [3]:
# 3) Cocokkan dengan EV kasar di EDA (sql/03_eda.sql, Q6)
seg = (v.groupby(["tenure_bucket", "contract"])
         .agg(n=("ev", "size"), churn=("churned", "mean"), p_rata=("p_churn", "mean"), ev_rata=("ev", "mean"))
         .round(3).sort_values("ev_rata", ascending=False))
print(seg)

                                 n  churn  p_rata    ev_rata
tenure_bucket contract                                      
0-5           Month-to-month  1318  0.561   0.543  24263.549
6-11          Month-to-month   590  0.425   0.422  22419.080
12-23         Month-to-month   765  0.376   0.369  20232.961
24-47         Month-to-month   845  0.336   0.340  19904.806
48+           Month-to-month   357  0.258   0.309  19025.706
              One year         662  0.131   0.121   4591.501
24-47         One year         510  0.098   0.105   2203.055
12-23         One year         199  0.106   0.076   -861.300
0-5           One year          32  0.125   0.104  -1472.384
6-11          One year          70  0.057   0.082  -1622.405
48+           Two year        1284  0.033   0.033  -2640.665
0-5           Two year          21    0.0   0.062  -2860.218
24-47         Two year         269  0.019   0.034  -3145.323
6-11          Two year          38    0.0   0.035  -3997.020
12-23         Two year  

In [4]:
# 4) Siapa yang layak? Peluang pergi pelanggan vs titik impasnya
v["layak"] = v.ev > 0
print(v.groupby("contract")["layak"].agg(["size", "mean"]).round(3))
print(v.groupby("senior_citizen")["layak"].agg(["size", "mean"]).round(3))

                size   mean
contract                   
Month-to-month  3875  0.857
One year        1473  0.448
Two year        1695  0.127
                size   mean
senior_citizen             
0               5901  0.549
1               1142  0.838
